In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')


RESULTS_DIR = Path('phase2/results/20260410_003735/')
print(f'Loading results from {RESULTS_DIR}...')

In [ ]:
# # Load all metrics files
# metrics_files = {
#     'uniform': 'uniform/20260413_194452/metrics.json',
#     'contribution': 'contribution/20260413_195056/metrics.json',
#     'contribution_oracle': 'contribution_oracle/20260413_194804/metrics.json',
#     'counterfactual_contribution': 'counterfactual_contribution/20260413_194814/metrics.json',
#     'hybrid': 'hybrid/20260413_201229/metrics.json',
#     'stake': 'stake/20260413_194416/metrics.json',
#     'bid_to_speak': 'bid_to_speak/20260413_194423/metrics.json',
#     'free_debate': 'free_debate/20260413_200000/metrics.json',
#     'forced_sharing': 'forced_sharing/20260413_194852/metrics.json',
#     'no_comm': 'no_comm/20260413_194857/metrics.json',
# }

# Your single merged metrics file
METRICS_FILE = 'metrics.json'  # adjust path as needed

metrics_files = {name: METRICS_FILE for name in [
    'free_debate',
    'contribution',
    'forced_sharing',
    'contribution_oracle',
    'counterfactual_contribution',
    'no_comm',
    # 'hybrid',
    'bid_to_speak',
    'uniform',
    'stake',
]}

all_metrics = {}
for name, path in metrics_files.items():
    with open(RESULTS_DIR / path) as f:
        data = json.load(f)
        # Handle nested structure (metrics are inside incentive key)
        all_metrics[name] = data.get(name, data)

print(f"Loaded {len(all_metrics)} incentive mechanisms")
print("Mechanisms:", list(all_metrics.keys()))

In [ ]:
# Create summary dataframe
summary_data = []
for name, m in all_metrics.items():
    summary_data.append({
        'Incentive': name,
        'Accuracy': m.get('accuracy', np.nan),
        'Accuracy Std': m.get('std_accuracy', np.nan),
        'Decisive Surfacing Rate': m.get('decisive_surfacing_rate', np.nan),
        'Free Riding Rate': m.get('free_riding_rate', np.nan),
        'Novelty Rate': m.get('novelty_rate', np.nan),
        'Mean Disclosure Cost': m.get('mean_disclosure_cost', np.nan),
        'Mean Comm Tokens': m.get('mean_communication_tokens', np.nan),
        'Time to Decisive': m.get('time_to_decisive_surfacing', np.nan),
    })

df = pd.DataFrame(summary_data)
df = df.set_index('Incentive')
# df = df.sort_values('Accuracy', ascending=False)
df.round(3)

## Key Metrics Comparison

In [ ]:
# Accuracy comparison with confidence intervals
fig, ax = plt.subplots(figsize=(12, 6))

incentives = df.index.tolist()
accuracies = df['Accuracy'].values
stds = df['Accuracy Std'].values

# 95% CI assuming 180 samples (60 scenarios × 3 runs)
n = 180
ci = 1.96 * stds / np.sqrt(n)

colors = sns.color_palette('husl', len(incentives))
bars = ax.bar(range(len(incentives)), accuracies, yerr=ci, capsize=5, color=colors, edgecolor='black')

ax.set_xticks(range(len(incentives)))
ax.set_xticklabels(incentives, rotation=45, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Group Decision Accuracy by Incentive Mechanism (temp=0.0)')
ax.set_ylim(0, 1)
ax.axhline(y=accuracies.mean(), color='red', linestyle='--', label=f'Mean: {accuracies.mean():.3f}')
ax.legend()

plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Decisive Feature Surfacing Rate
fig, ax = plt.subplots(figsize=(12, 6))

surfacing_rates = df['Decisive Surfacing Rate'].values
colors = sns.color_palette('husl', len(incentives))

bars = ax.bar(range(len(incentives)), surfacing_rates, color=colors, edgecolor='black')

ax.set_xticks(range(len(incentives)))
ax.set_xticklabels(incentives, rotation=45, ha='right')
ax.set_ylabel('Decisive Surfacing Rate')
ax.set_title('Rate of Decisive Feature Disclosure by Incentive Mechanism')
ax.set_ylim(0, 1)
ax.axhline(y=surfacing_rates.mean(), color='red', linestyle='--', label=f'Mean: {surfacing_rates.mean():.3f}')
ax.legend()

plt.tight_layout()
plt.savefig('surfacing_rate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Free Riding Rate (lower is better for cooperation)
fig, ax = plt.subplots(figsize=(12, 6))

free_riding = df['Free Riding Rate'].values
colors = sns.color_palette('husl', len(incentives))

bars = ax.bar(range(len(incentives)), free_riding, color=colors, edgecolor='black')

ax.set_xticks(range(len(incentives)))
ax.set_xticklabels(incentives, rotation=45, ha='right')
ax.set_ylabel('Free Riding Rate')
ax.set_title('Free Riding Rate by Incentive Mechanism (Lower = More Cooperation)')
ax.set_ylim(0, 1)
ax.axhline(y=free_riding.mean(), color='red', linestyle='--', label=f'Mean: {free_riding.mean():.3f}')
ax.legend()

plt.tight_layout()
plt.savefig('free_riding_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Multi-Metric Radar Chart

In [ ]:
# Radar chart for top 5 mechanisms
from math import pi

# Select metrics and normalize
metrics_for_radar = ['Accuracy', 'Decisive Surfacing Rate', 'Novelty Rate']
# Invert free riding (1 - rate) so higher is better
df_radar = df[metrics_for_radar].copy()
df_radar['Cooperation Rate'] = 1 - df['Free Riding Rate']

# Normalize to 0-1
df_radar_norm = (df_radar - df_radar.min()) / (df_radar.max() - df_radar.min())

# Top 5 by accuracy
top5 = df_radar_norm.head(5)

categories = list(top5.columns)
N = len(categories)

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for i, (idx, row) in enumerate(top5.iterrows()):
    values = row.values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=idx)
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_title('Top 5 Incentive Mechanisms - Multi-Metric Comparison', size=14, y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.savefig('radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Statistical Tests

In [ ]:
# Compare top mechanism vs baseline (uniform)
baseline = 'uniform'
top_mech = df.index[0]  # highest accuracy

print(f"Comparing {top_mech} vs {baseline}")
print(f"\n{top_mech}:")
print(f"  Accuracy: {all_metrics[top_mech]['accuracy']:.3f} ± {all_metrics[top_mech]['std_accuracy']:.3f}")
print(f"\n{baseline}:")
print(f"  Accuracy: {all_metrics[baseline]['accuracy']:.3f} ± {all_metrics[baseline]['std_accuracy']:.3f}")

# Cohen's d effect size (using pooled std)
m1, s1 = all_metrics[top_mech]['accuracy'], all_metrics[top_mech]['std_accuracy']
m2, s2 = all_metrics[baseline]['accuracy'], all_metrics[baseline]['std_accuracy']
pooled_std = np.sqrt((s1**2 + s2**2) / 2)
cohens_d = (m1 - m2) / pooled_std if pooled_std > 0 else 0

print(f"\nEffect Size (Cohen's d): {cohens_d:.3f}")
if abs(cohens_d) < 0.2:
    print("  Interpretation: Negligible effect")
elif abs(cohens_d) < 0.5:
    print("  Interpretation: Small effect")
elif abs(cohens_d) < 0.8:
    print("  Interpretation: Medium effect")
else:
    print("  Interpretation: Large effect")

In [ ]:
# Summary statistics table
print("\n" + "="*80)
print("SUMMARY: INCENTIVE MECHANISM COMPARISON (Temperature = 0.0)")
print("="*80)
print(f"\nBest Accuracy: {df.index[0]} ({df['Accuracy'].iloc[0]:.1%})")
print(f"Worst Accuracy: {df.index[-1]} ({df['Accuracy'].iloc[-1]:.1%})")
print(f"\nHighest Surfacing Rate: {df.sort_values('Decisive Surfacing Rate', ascending=False).index[0]}")
print(f"Lowest Free Riding: {df.sort_values('Free Riding Rate').index[0]}")

print("\n" + "-"*80)
print(df[['Accuracy', 'Decisive Surfacing Rate', 'Free Riding Rate']].round(3).to_string())

## Domain-Specific Analysis

In [ ]:
# Domain accuracy heatmap
domain_data = {}
for name, m in all_metrics.items():
    if 'per_domain' in m:
        domain_data[name] = m['per_domain']

if domain_data:
    domain_df = pd.DataFrame(domain_data)
    
    # Sort by mean accuracy across mechanisms
    domain_df['mean'] = domain_df.mean(axis=1)
    domain_df = domain_df.sort_values('mean', ascending=False)
    domain_df = domain_df.drop('mean', axis=1)
    
    plt.figure(figsize=(14, 16))
    sns.heatmap(domain_df, annot=True, fmt='.2f', cmap='RdYlGn', 
                vmin=0, vmax=1, linewidths=0.5)
    plt.title('Accuracy by Domain and Incentive Mechanism')
    plt.xlabel('Incentive Mechanism')
    plt.ylabel('Domain')
    plt.tight_layout()
    plt.savefig('domain_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No per-domain data available")

In [ ]:
# Save summary to CSV
df.to_csv('incentive_comparison_summary.csv')
print("Summary saved to incentive_comparison_summary.csv")

## Core Research Questions

### 1. Free-Rider Problem: Does Uniform Reward Lead to Under-Disclosure?
The central hypothesis: uniform incentives + costly disclosure = rational under-revelation of decisive information.

In [ ]:
# Key comparison: Uniform vs Contribution-based incentives
# This tests the core hypothesis about free-riding

uniform_acc = all_metrics['uniform']['accuracy']
contribution_acc = all_metrics['contribution']['accuracy']
contribution_oracle_acc = all_metrics['contribution_oracle']['accuracy']

uniform_surfacing = all_metrics['uniform']['decisive_surfacing_rate']
contribution_surfacing = all_metrics['contribution']['decisive_surfacing_rate']
contribution_oracle_surfacing = all_metrics['contribution_oracle']['decisive_surfacing_rate']

print("="*70)
print("HYPOTHESIS TEST: Do contribution-based incentives reduce free-riding?")
print("="*70)

print(f"\n{'Metric':<30} {'Uniform':<15} {'Contribution':<15} {'Oracle':<15}")
print("-"*70)
print(f"{'Accuracy':<30} {uniform_acc:.3f}          {contribution_acc:.3f}          {contribution_oracle_acc:.3f}")
print(f"{'Decisive Surfacing Rate':<30} {uniform_surfacing:.3f}          {contribution_surfacing:.3f}          {contribution_oracle_surfacing:.3f}")
print(f"{'Free Riding Rate':<30} {all_metrics['uniform']['free_riding_rate']:.3f}          {all_metrics['contribution']['free_riding_rate']:.3f}          {all_metrics['contribution_oracle']['free_riding_rate']:.3f}")

# Improvement percentages
acc_improvement = (contribution_acc - uniform_acc) / uniform_acc * 100
surfacing_improvement = (contribution_surfacing - uniform_surfacing) / uniform_surfacing * 100 if uniform_surfacing > 0 else 0

print(f"\n📈 Contribution vs Uniform:")
print(f"   Accuracy improvement: {acc_improvement:+.1f}%")
print(f"   Surfacing improvement: {surfacing_improvement:+.1f}%")

### 2. Disclosure Calibration: Are Agents Cost-Sensitive?
Do agents share high-value features and withhold noise? Or do they free-ride?

In [ ]:
# Surfacing rate by feature cost (from metrics)
# This shows if high-cost decisive features are under-disclosed

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Surfacing by cost for each mechanism
ax1 = axes[0]
for name, m in all_metrics.items():
    if 'surfacing_by_cost' in m and m['surfacing_by_cost']:
        costs = sorted([int(k) for k in m['surfacing_by_cost'].keys()])
        rates = [m['surfacing_by_cost'][str(c)] for c in costs]
        ax1.plot(costs, rates, 'o-', label=name, alpha=0.7)

ax1.set_xlabel('Feature Disclosure Cost')
ax1.set_ylabel('Surfacing Rate')
ax1.set_title('Disclosure Rate vs Feature Cost\n(Lower surfacing at high cost = cost-sensitivity)')
ax1.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax1.set_xticks([1, 2, 3, 4, 5])
ax1.grid(True, alpha=0.3)

# Plot 2: Selective Disclosure Index comparison
ax2 = axes[1]
sdi_data = [(name, m.get('selective_disclosure_index', 0)) for name, m in all_metrics.items()]
sdi_data.sort(key=lambda x: x[1], reverse=True)
names, sdis = zip(*sdi_data)

colors = sns.color_palette('husl', len(names))
ax2.barh(range(len(names)), sdis, color=colors)
ax2.set_yticks(range(len(names)))
ax2.set_yticklabels(names)
ax2.set_xlabel('Selective Disclosure Index')
ax2.set_title('Selective Disclosure Index\n(Higher = More focused on decisive features)')
ax2.axvline(x=np.mean(sdis), color='red', linestyle='--', label=f'Mean: {np.mean(sdis):.3f}')

plt.tight_layout()
plt.savefig('disclosure_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

### 3. Misleading Feature Influence
How often do low-cost misleading features dominate discussion and steer the group wrong?

In [ ]:
# Misleading feature analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Misleading before wrong rate
misleading_data = []
for name, m in all_metrics.items():
    misleading_data.append({
        'Incentive': name,
        'Misleading Before Wrong': m.get('misleading_before_wrong_rate', 0),
        'Misleading Preceded Decisive': m.get('misleading_preceded_decisive_rate', 0),
    })

misleading_df = pd.DataFrame(misleading_data).set_index('Incentive')
misleading_df = misleading_df.sort_values('Misleading Before Wrong', ascending=False)

ax1 = axes[0]
x = np.arange(len(misleading_df))
width = 0.35
bars1 = ax1.bar(x - width/2, misleading_df['Misleading Before Wrong'], width, label='Misleading Before Wrong', color='coral')
bars2 = ax1.bar(x + width/2, misleading_df['Misleading Preceded Decisive'], width, label='Misleading Preceded Decisive', color='steelblue')

ax1.set_ylabel('Rate')
ax1.set_title('Misleading Feature Influence\n(Lower = Better noise filtering)')
ax1.set_xticks(x)
ax1.set_xticklabels(misleading_df.index, rotation=45, ha='right')
ax1.legend()
ax1.set_ylim(0, max(misleading_df.max()) * 1.2)

# Plot 2: Decisive holder disclosure rate vs non-holder
ax2 = axes[1]
holder_data = []
for name, m in all_metrics.items():
    holder_data.append({
        'Incentive': name,
        'Decisive Holder Rate': m.get('decisive_holder_disclosure_rate', 0),
        'Non-Holder Rate': m.get('non_holder_disclosure_rate', 0),
    })

holder_df = pd.DataFrame(holder_data).set_index('Incentive')
holder_df = holder_df.sort_values('Decisive Holder Rate', ascending=False)

x = np.arange(len(holder_df))
bars1 = ax2.bar(x - width/2, holder_df['Decisive Holder Rate'], width, label='Decisive Holder', color='green')
bars2 = ax2.bar(x + width/2, holder_df['Non-Holder Rate'], width, label='Non-Holder', color='gray')

ax2.set_ylabel('Disclosure Rate')
ax2.set_title('Who Discloses?\n(Decisive holders should disclose more)')
ax2.set_xticks(x)
ax2.set_xticklabels(holder_df.index, rotation=45, ha='right')
ax2.legend()

plt.tight_layout()
plt.savefig('misleading_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 4. Cost-Effectiveness Analysis
Accuracy per communication cost - mapping the accuracy-cost frontier

In [ ]:
# Cost-effectiveness: Accuracy vs Communication Cost
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gather data
cost_eff_data = []
for name, m in all_metrics.items():
    acc = m.get('accuracy', 0)
    tokens = m.get('mean_communication_tokens', 1)
    disclosure_cost = m.get('mean_disclosure_cost', 0)
    cost_eff_data.append({
        'Incentive': name,
        'Accuracy': acc,
        'Mean Tokens': tokens,
        'Mean Disclosure Cost': disclosure_cost,
        'Accuracy per 1K Tokens': acc / (tokens / 1000) if tokens > 0 else 0,
    })

cost_df = pd.DataFrame(cost_eff_data)

# Plot 1: Accuracy vs Token Cost (Pareto frontier)
ax1 = axes[0]
colors = sns.color_palette('husl', len(cost_df))
for i, row in cost_df.iterrows():
    ax1.scatter(row['Mean Tokens'], row['Accuracy'], s=150, c=[colors[i]], edgecolor='black', zorder=5)
    ax1.annotate(row['Incentive'], (row['Mean Tokens'], row['Accuracy']), 
                 textcoords="offset points", xytext=(5, 5), fontsize=8)

ax1.set_xlabel('Mean Communication Tokens')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy-Cost Frontier\n(Upper-left = most efficient)')
ax1.grid(True, alpha=0.3)

# Highlight Pareto-optimal points
sorted_df = cost_df.sort_values('Mean Tokens')
pareto_acc = 0
pareto_points = []
for _, row in sorted_df.iterrows():
    if row['Accuracy'] > pareto_acc:
        pareto_points.append((row['Mean Tokens'], row['Accuracy']))
        pareto_acc = row['Accuracy']

if pareto_points:
    px, py = zip(*pareto_points)
    ax1.plot(px, py, 'r--', alpha=0.5, label='Pareto frontier')
    ax1.legend()

# Plot 2: Cost-effectiveness ranking
ax2 = axes[1]
cost_df_sorted = cost_df.sort_values('Accuracy per 1K Tokens', ascending=True)
colors = sns.color_palette('husl', len(cost_df_sorted))
ax2.barh(range(len(cost_df_sorted)), cost_df_sorted['Accuracy per 1K Tokens'], color=colors)
ax2.set_yticks(range(len(cost_df_sorted)))
ax2.set_yticklabels(cost_df_sorted['Incentive'])
ax2.set_xlabel('Accuracy per 1K Tokens')
ax2.set_title('Cost-Effectiveness Ranking\n(Higher = more efficient)')

plt.tight_layout()
plt.savefig('cost_effectiveness.png', dpi=150, bbox_inches='tight')
plt.show()

# Print cost-effectiveness table
print("\nCost-Effectiveness Summary:")
print(cost_df.sort_values('Accuracy per 1K Tokens', ascending=False).to_string(index=False))

### 5. Time to Decisive Information
How quickly does decisive information surface? Earlier is better.

In [ ]:
# Time to decisive surfacing
time_data = [(name, m.get('time_to_decisive_surfacing', np.nan)) for name, m in all_metrics.items()]
time_df = pd.DataFrame(time_data, columns=['Incentive', 'Time to Decisive'])
time_df = time_df.dropna().sort_values('Time to Decisive')

fig, ax = plt.subplots(figsize=(12, 5))
colors = sns.color_palette('husl', len(time_df))
bars = ax.barh(range(len(time_df)), time_df['Time to Decisive'], color=colors)
ax.set_yticks(range(len(time_df)))
ax.set_yticklabels(time_df['Incentive'])
ax.set_xlabel('Mean Round When Decisive Feature Surfaces')
ax.set_title('Time to Decisive Information Surfacing\n(Lower = faster revelation)')
ax.axvline(x=time_df['Time to Decisive'].mean(), color='red', linestyle='--', 
           label=f"Mean: {time_df['Time to Decisive'].mean():.2f}")
ax.legend()

plt.tight_layout()
plt.savefig('time_to_decisive.png', dpi=150, bbox_inches='tight')
plt.show()

### 6. Mechanism Categories Comparison
Group mechanisms by type: Market-based, Contribution-based, Baselines

In [ ]:
# Group mechanisms by category
mechanism_categories = {
    'Baseline': ['uniform', 'free_debate', 'no_comm'],
    'Contribution-Based': ['contribution', 'contribution_oracle', 'counterfactual_contribution'],
    'Market-Based': ['stake', 'bid_to_speak'],
    'Hybrid/Forced': ['hybrid', 'forced_sharing'],
}

category_stats = []
for cat, mechs in mechanism_categories.items():
    cat_accs = [all_metrics[m]['accuracy'] for m in mechs if m in all_metrics]
    cat_surfacing = [all_metrics[m]['decisive_surfacing_rate'] for m in mechs if m in all_metrics]
    cat_free_riding = [all_metrics[m]['free_riding_rate'] for m in mechs if m in all_metrics]
    
    category_stats.append({
        'Category': cat,
        'Mean Accuracy': np.mean(cat_accs),
        'Std Accuracy': np.std(cat_accs),
        'Mean Surfacing': np.mean(cat_surfacing),
        'Mean Free Riding': np.mean(cat_free_riding),
        'N': len(cat_accs),
    })

cat_df = pd.DataFrame(category_stats)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy by category
ax1 = axes[0]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4']
bars = ax1.bar(cat_df['Category'], cat_df['Mean Accuracy'], yerr=cat_df['Std Accuracy'], 
               capsize=5, color=colors, edgecolor='black')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy by Mechanism Category')
ax1.set_ylim(0, 1)
ax1.tick_params(axis='x', rotation=15)

# Surfacing by category
ax2 = axes[1]
ax2.bar(cat_df['Category'], cat_df['Mean Surfacing'], color=colors, edgecolor='black')
ax2.set_ylabel('Decisive Surfacing Rate')
ax2.set_title('Decisive Surfacing by Category')
ax2.set_ylim(0, 1)
ax2.tick_params(axis='x', rotation=15)

# Free riding by category
ax3 = axes[2]
ax3.bar(cat_df['Category'], cat_df['Mean Free Riding'], color=colors, edgecolor='black')
ax3.set_ylabel('Free Riding Rate')
ax3.set_title('Free Riding by Category')
ax3.set_ylim(0, 1)
ax3.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('category_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCategory Summary:")
print(cat_df.round(3).to_string(index=False))

### 7. Final Paper-Ready Summary Table

In [ ]:
# Paper-ready LaTeX table
print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Incentive Mechanism Comparison (Temperature = 0.0, N=180 per mechanism)}")
print("\\begin{tabular}{lcccccc}")
print("\\toprule")
print("Mechanism & Accuracy & Surfacing & Free-Riding & SDI & Time & Tokens \\\\")
print("\\midrule")

for name in df.index:
    m = all_metrics[name]
    acc = m.get('accuracy', 0)
    surf = m.get('decisive_surfacing_rate', 0)
    free = m.get('free_riding_rate', 0)
    sdi = m.get('selective_disclosure_index', 0)
    time_val = m.get('time_to_decisive_surfacing', 0)
    tokens = m.get('mean_communication_tokens', 0)
    
    escaped_name = name.replace('_', '\\_')
    print(f"{escaped_name} & {acc:.3f} & {surf:.3f} & {free:.3f} & {sdi:.3f} & {time_val:.2f} & {tokens:.0f} \\\\")

print("\\bottomrule")
print("\\end{tabular}")
print("\\label{tab:incentive_comparison}")
print("\\end{table}")

print("\n" + "="*80)
print("KEY FINDINGS:")
print("="*80)
print(f"\n1. Best Accuracy: {df.index[0]} ({df['Accuracy'].iloc[0]:.1%})")
print(f"2. Best Surfacing: {df.sort_values('Decisive Surfacing Rate', ascending=False).index[0]}")
print(f"3. Lowest Free-Riding: {df.sort_values('Free Riding Rate').index[0]}")
print(f"\n4. Contribution-based vs Uniform improvement:")
acc_diff = (all_metrics['contribution']['accuracy'] - all_metrics['uniform']['accuracy'])*100
surf_diff = (all_metrics['contribution']['decisive_surfacing_rate'] - all_metrics['uniform']['decisive_surfacing_rate'])*100
print(f"   Accuracy: {acc_diff:+.1f}pp")
print(f"   Surfacing: {surf_diff:+.1f}pp")